# Level 3: Data Exploration, Visualization, and Prediction

This notebook explores the train-level dataset created in Level 2. It compares journey durations, measures station traffic, visualizes the distance relationship, and trains a linear regression model using distance and number of stops.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

journeys = pd.read_csv('../data/train_level2_cleaned.csv')
journeys['Journey_Duration'] = pd.to_timedelta(journeys['Journey_Duration'])
journeys['Journey_Hours'] = journeys['Journey_Duration'].dt.total_seconds() / 3600
journeys.head()

In [ ]:
summary = journeys[['Total_Distance', 'Number_of_Stops', 'Journey_Hours']].describe().T
display(summary)
print(f'Average journey duration: {journeys.Journey_Hours.mean():.2f} hours')
print(f'Median journey duration: {journeys.Journey_Hours.median():.2f} hours')
print(f'Distance-duration correlation: {journeys.Total_Distance.corr(journeys.Journey_Hours):.4f}')

## Journey duration across routes

In [ ]:
route_duration = journeys.groupby(['Start_Station', 'End_Station'], as_index=False).agg(
    Average_Journey_Hours=('Journey_Hours', 'mean'),
    Journeys=('Train_No', 'count'),
)
route_duration = route_duration.sort_values('Average_Journey_Hours', ascending=False).head(15)
route_duration['Route'] = route_duration['Start_Station'] + ' -> ' + route_duration['End_Station']
plt.figure(figsize=(10, 6))
sns.barplot(data=route_duration, y='Route', x='Average_Journey_Hours', color='steelblue')
plt.title('Routes with the longest average journey duration')
plt.xlabel('Average journey duration (hours)')
plt.ylabel('Route')
plt.tight_layout()
plt.show()

## Station-wise train traffic

In [ ]:
station_traffic = pd.concat([
    journeys['Start_Station'].rename('Station'),
    journeys['End_Station'].rename('Station')
]).value_counts().rename_axis('Station').reset_index(name='Train_Associations')
display(station_traffic.head(10))
plt.figure(figsize=(10, 6))
sns.barplot(data=station_traffic.head(15).sort_values('Train_Associations'), y='Station', x='Train_Associations', color='darkorange')
plt.title('Top stations by train associations')
plt.tight_layout()
plt.show()

## Distance and duration relationship

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=journeys, x='Total_Distance', y='Journey_Hours', alpha=0.35)
sns.regplot(data=journeys, x='Total_Distance', y='Journey_Hours', scatter=False, color='red')
plt.title('Total distance vs journey duration')
plt.xlabel('Total distance (km)')
plt.ylabel('Journey duration (hours)')
plt.tight_layout()
plt.show()
print(f"Correlation: {journeys['Total_Distance'].corr(journeys['Journey_Hours']):.4f}")

## Linear regression prediction model

Use `Total_Distance` and `Number_of_Stops` to predict `Journey_Hours`. The split uses a fixed random seed so the evaluation is reproducible.

In [ ]:
features = ['Total_Distance', 'Number_of_Stops']
X = journeys[features]
y = journeys['Journey_Hours']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = LinearRegression()
model.fit(X_train, y_train)
predictions = model.predict(X_test)
metrics = {
    'MAE (hours)': mean_absolute_error(y_test, predictions),
    'RMSE (hours)': np.sqrt(mean_squared_error(y_test, predictions)),
    'R2 score': r2_score(y_test, predictions),
}
display(pd.Series(metrics))
print('Intercept:', model.intercept_)
print('Coefficients:', dict(zip(features, model.coef_)))

In [ ]:
def predict_journey_hours(total_distance, number_of_stops):
    values = pd.DataFrame([[total_distance, number_of_stops]], columns=features)
    return float(model.predict(values)[0])

example_hours = predict_journey_hours(500, 12)
print(f'Predicted duration for 500 km and 12 stops: {example_hours:.2f} hours')